# Hack-A-Roo 2026, AI for Quantum track
# Notebook 2: Readout error classifier

**Problem.** Every measurement on a quantum computer is a noisy readout. A qubit in state 0 is sometimes reported as 1 and vice versa. The standard fix is to measure every basis state, tabulate a confusion matrix from the counts, and invert it. That table has 2^n columns, each one a distribution estimated from a handful of shots. A model of the readout noise with fewer parameters can be estimated from the same shots with less variance, if the model is right. Which model of the noise gives the best mitigation at a small calibration budget?

**What this notebook already does.** It builds an independent per-qubit readout noise model, runs calibration circuits that prepare every basis state and measure it, and builds three models of the noise from the same calibration shots: the raw confusion matrix (240 free parameters), a logistic-regression classifier from the prepared bits to the observed state (75 free parameters), and an independent per-qubit model estimated by pooling the shots qubit by qubit (8 free parameters). All three are inverted the same way to mitigate test circuits whose ideal output distribution is known.

**Metric.** Total variation distance between the mitigated distribution and the ideal distribution on test circuits. Lower is better. Zero means perfect mitigation. Each model's mean absolute entry error against the true noise matrix is printed too, since the true matrix is known in a simulation.

**Hypothesis to test.** At a small calibration budget, the model with the fewest parameters wins when its structure matches the noise, and the raw table catches up as the budget grows. The notebook does not assume this. One calibration realization and three test circuits do not establish a general ranking, and the poster should repeat the seeds before claiming one.

**Deliverable for the poster.** One bar chart comparing unmitigated and the three models on each test circuit, and one number per model (mean total variation distance). Then whichever shred-out you pick at the bottom.

Run every cell top to bottom once. The base run takes under a minute. Then go to the Shred-outs section.

In [ ]:
# CONFIG CELL. Every constant, seed, and path lives here. Nothing is hardcoded downstream.

RANDOM_SEED = 42

# System
QUBIT_COUNT = 4                     # calibration needs 2^QUBIT_COUNT circuits, so keep this small in the base run

# Readout noise. Each qubit draws its own error rates from these ranges. The qubits are INDEPENDENT in the base run.
PROBABILITY_READ_1_GIVEN_0_LOW = 0.02
PROBABILITY_READ_1_GIVEN_0_HIGH = 0.08
PROBABILITY_READ_0_GIVEN_1_LOW = 0.05   # 1 -> 0 errors are usually worse on real hardware (relaxation during readout)
PROBABILITY_READ_0_GIVEN_1_HIGH = 0.15

# Shots
CALIBRATION_SHOTS_PER_STATE = 50    # deliberately small. The raw table is noisy here; see shred-out 4 for the sweep.
TEST_SHOTS = 4000

# Classifier
LOGISTIC_REGRESSION_MAX_ITERATIONS = 2000
LOGISTIC_REGRESSION_REGULARIZATION = 100.0   # sklearn's C. Larger is less regularization; sklearn's default of 1.0 biases the matrix noticeably.

# Output
OUTPUT_PATH = "readout_classifier_dataset.pkl"
FIGURE_PATH = "readout_classifier_figure.png"

In [ ]:
# Install. Skip this cell if your environment already has these. Tested with qiskit 2.6, qiskit-aer 0.18, scikit-learn 1.8 and 1.9.
!pip install -q "qiskit[visualization]" "qiskit-aer>=0.17" "scikit-learn>=1.5" matplotlib

In [ ]:
import time
import pickle
import numpy as np
import sklearn
import qiskit
import qiskit_aer
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError
from sklearn.linear_model import LogisticRegression

random_generator = np.random.default_rng(RANDOM_SEED)
NUMBER_OF_BASIS_STATES = 2 ** QUBIT_COUNT
package_versions = {"numpy": np.__version__, "qiskit": qiskit.__version__, "qiskit-aer": qiskit_aer.__version__,
                    "scikit-learn": sklearn.__version__}
print("imports done,", NUMBER_OF_BASIS_STATES, "basis states", package_versions)

In [ ]:
# Build the readout noise model. One independent error matrix per qubit.
#
# For qubit q the readout error matrix is
#   [[ P(read 0 | true 0), P(read 1 | true 0) ],
#    [ P(read 0 | true 1), P(read 1 | true 1) ]]
# Rows are the true state, columns are the reported state. Rows sum to 1.
#
# This base model is UNCORRELATED: each qubit flips independently. Real devices have correlated readout errors
# between neighboring qubits. Simulating those is shred-out 1, and it is NOT done by adding a multi-qubit
# ReadoutError to this noise model: Aer attaches readout errors to measure instructions by their qubits, and
# this circuit measures each qubit with its own one-qubit instruction, so a two-qubit error added on [0, 1] is
# accepted and then never applied (checked in Aer 0.18: a deterministic 00 -> 11 table returned 00 on every shot).

per_qubit_probability_read_1_given_0 = random_generator.uniform(
    PROBABILITY_READ_1_GIVEN_0_LOW, PROBABILITY_READ_1_GIVEN_0_HIGH, size=QUBIT_COUNT)
per_qubit_probability_read_0_given_1 = random_generator.uniform(
    PROBABILITY_READ_0_GIVEN_1_LOW, PROBABILITY_READ_0_GIVEN_1_HIGH, size=QUBIT_COUNT)

noise_model = NoiseModel()
for qubit_index in range(QUBIT_COUNT):
    p10 = per_qubit_probability_read_1_given_0[qubit_index]
    p01 = per_qubit_probability_read_0_given_1[qubit_index]
    error_matrix = [[1 - p10, p10], [p01, 1 - p01]]
    noise_model.add_readout_error(ReadoutError(error_matrix), [qubit_index])

noisy_simulator = AerSimulator(noise_model=noise_model, seed_simulator=RANDOM_SEED)

print("true per-qubit error rates (hidden from the mitigators, they only see shots):")
for qubit_index in range(QUBIT_COUNT):
    print(f"  qubit {qubit_index}: P(read 1 | true 0) = {per_qubit_probability_read_1_given_0[qubit_index]:.3f}"
          f"   P(read 0 | true 1) = {per_qubit_probability_read_0_given_1[qubit_index]:.3f}")

In [ ]:
# Helpers for turning qiskit counts into probability vectors indexed the same way as the statevector.
#
# qiskit reports a bitstring like '0110' with qubit 0 as the RIGHTMOST character.
# The statevector index for that bitstring is int('0110', 2) = 6, which also treats qubit 0 as the
# least significant bit. So int(bitstring, 2) is the right index and nothing needs reversing.

def counts_to_probability_vector(counts, shots):
    """Turns a qiskit counts dict into a length 2^n probability vector"""
    probability_vector = np.zeros(NUMBER_OF_BASIS_STATES)
    for bitstring, count in counts.items():
        probability_vector[int(bitstring, 2)] = count / shots
    assert abs(probability_vector.sum() - 1) < 1e-9
    return probability_vector


def index_to_bits(state_index):
    """Turns a basis state index into a length n array of bits, bit q is qubit q"""
    return np.array([(state_index >> qubit_index) & 1 for qubit_index in range(QUBIT_COUNT)])


def total_variation_distance(distribution_a, distribution_b):
    """Half the L1 distance between two probability vectors"""
    assert distribution_a.shape == distribution_b.shape
    return 0.5 * np.abs(distribution_a - distribution_b).sum()


# Table of every basis state's bits, used by all three noise models below. every_state_bits[state, qubit].
every_state_bits = np.array([index_to_bits(state_index) for state_index in range(NUMBER_OF_BASIS_STATES)])
assert every_state_bits.shape == (NUMBER_OF_BASIS_STATES, QUBIT_COUNT)
print("bits of state index 6:", index_to_bits(6))

In [ ]:
# Calibration: prepare each of the 2^n basis states with X gates, measure it through the noisy readout.
# This is what the mitigators learn from. Nothing else about the noise is given to them.

calibration_circuits = []
for prepared_state_index in range(NUMBER_OF_BASIS_STATES):
    circuit = QuantumCircuit(QUBIT_COUNT, QUBIT_COUNT)
    for qubit_index in range(QUBIT_COUNT):
        if every_state_bits[prepared_state_index, qubit_index] == 1:
            circuit.x(qubit_index)
    circuit.measure(range(QUBIT_COUNT), range(QUBIT_COUNT))
    calibration_circuits.append(circuit)

start_time = time.time()
calibration_job = noisy_simulator.run(calibration_circuits, shots=CALIBRATION_SHOTS_PER_STATE)
calibration_counts = calibration_job.result().get_counts()
print(f"ran {NUMBER_OF_BASIS_STATES} calibration circuits in {time.time() - start_time:.1f}s")

# Noise model 1, the raw table. confusion_matrix[observed, prepared] = fraction of shots on the prepared state that
# were read as observed. 16 columns that each sum to 1, so 16 * 15 = 240 free parameters.
confusion_matrix = np.zeros((NUMBER_OF_BASIS_STATES, NUMBER_OF_BASIS_STATES))
for prepared_state_index in range(NUMBER_OF_BASIS_STATES):
    confusion_matrix[:, prepared_state_index] = counts_to_probability_vector(
        calibration_counts[prepared_state_index], CALIBRATION_SHOTS_PER_STATE)
assert np.allclose(confusion_matrix.sum(axis=0), 1)

print("diagonal of the raw table (probability each state is read correctly):")
print(np.round(np.diag(confusion_matrix), 3))
print(f"example: prepared state {NUMBER_OF_BASIS_STATES - 1} (all ones) was read as:", calibration_counts[NUMBER_OF_BASIS_STATES - 1])

In [ ]:
# All three mitigators share one inversion step and differ only in the model of the noise they invert.
#   observed_distribution = noise_matrix @ true_distribution,   noise_matrix[observed, prepared] = P(observed | prepared)
# Solve for the true distribution by least squares, clip negatives, renormalize. That is the standard approach.

def mitigate_by_inversion(observed_distribution, noise_matrix):
    """Least-squares unfolding of an observed distribution through a noise matrix"""
    estimated_true, residuals, rank, singular_values = np.linalg.lstsq(noise_matrix, observed_distribution, rcond=None)
    estimated_true = np.clip(estimated_true, 0, None)
    assert estimated_true.sum() > 0, "every unfolded entry was clipped to zero; the noise matrix or the observation is degenerate"
    return estimated_true / estimated_true.sum()


# Noise model 2, the thing this notebook is about: a classifier from the PREPARED bits to the OBSERVED state.
# Multinomial logistic regression on n input bits with 2^n output classes has (n + 1) * 2^n = 80 coefficients, of
# which 75 are free after the softmax gauge is removed. Its log-probabilities are linear in the prepared bits, which
# is exactly the form independent per-qubit readout errors have, so it CAN represent this noise model exactly but
# it is not told to. Correlated noise is not linear in the bits, and that is shred-out 1.
# Training rows are individual calibration shots: features = prepared bits, label = observed state index.

classifier_features = []
classifier_labels = []
for prepared_state_index in range(NUMBER_OF_BASIS_STATES):
    for observed_bitstring, count in calibration_counts[prepared_state_index].items():
        for shot_index in range(count):
            classifier_features.append(every_state_bits[prepared_state_index])
            classifier_labels.append(int(observed_bitstring, 2))
classifier_features = np.array(classifier_features)
classifier_labels = np.array(classifier_labels)
assert classifier_features.shape == (NUMBER_OF_BASIS_STATES * CALIBRATION_SHOTS_PER_STATE, QUBIT_COUNT)

readout_classifier = LogisticRegression(max_iter=LOGISTIC_REGRESSION_MAX_ITERATIONS, C=LOGISTIC_REGRESSION_REGULARIZATION)
start_time = time.time()
readout_classifier.fit(classifier_features, classifier_labels)
print(f"trained the classifier on {len(classifier_labels)} shots in {time.time() - start_time:.1f}s")
# Every observed state must appear at least once in the calibration shots or the classifier has no class for it.
assert list(readout_classifier.classes_) == list(range(NUMBER_OF_BASIS_STATES)), "some observed state never appeared; raise CALIBRATION_SHOTS_PER_STATE"

# learned_confusion_matrix[observed, prepared] = P(observed | prepared) according to the classifier
learned_confusion_matrix = readout_classifier.predict_proba(every_state_bits).T
assert learned_confusion_matrix.shape == (NUMBER_OF_BASIS_STATES, NUMBER_OF_BASIS_STATES)
assert np.allclose(learned_confusion_matrix.sum(axis=0), 1)


# Noise model 3, the independent-channel baseline: assume each qubit flips on its own, estimate its two error rates
# by pooling every calibration shot in which that qubit was prepared 0 (or 1), and take the tensor product.
# 2n = 8 free parameters. It uses no learned classifier. It is the right model for the noise in the base run and the
# wrong model for correlated noise, which is the whole point of comparing it.

pooled_probability_read_1_given_0 = np.zeros(QUBIT_COUNT)
pooled_probability_read_0_given_1 = np.zeros(QUBIT_COUNT)
for qubit_index in range(QUBIT_COUNT):
    prepared_0_rows = classifier_features[:, qubit_index] == 0
    observed_bits = every_state_bits[classifier_labels, qubit_index]
    pooled_probability_read_1_given_0[qubit_index] = observed_bits[prepared_0_rows].mean()
    pooled_probability_read_0_given_1[qubit_index] = 1 - observed_bits[~prepared_0_rows].mean()


def tensor_product_noise_matrix(probability_read_1_given_0, probability_read_0_given_1):
    """Full 2^n by 2^n noise matrix for independent per-qubit errors, in statevector index order"""
    full_matrix = np.ones((1, 1))
    for qubit_index in reversed(range(QUBIT_COUNT)):     # reversed so qubit 0 ends up as the least significant bit
        p10 = probability_read_1_given_0[qubit_index]
        p01 = probability_read_0_given_1[qubit_index]
        single_qubit_matrix = np.array([[1 - p10, p01], [p10, 1 - p01]])    # [observed, prepared]
        full_matrix = np.kron(full_matrix, single_qubit_matrix)
    assert np.allclose(full_matrix.sum(axis=0), 1)
    return full_matrix


independent_confusion_matrix = tensor_product_noise_matrix(pooled_probability_read_1_given_0, pooled_probability_read_0_given_1)
print("pooled per-qubit estimates, P(read 1 | true 0):", np.round(pooled_probability_read_1_given_0, 3))
print("pooled per-qubit estimates, P(read 0 | true 1):", np.round(pooled_probability_read_0_given_1, 3))

# How close is each model to the truth? The true matrix is built from the hidden rates. This number is for your
# understanding only; on hardware nobody has it.
true_confusion_matrix = tensor_product_noise_matrix(per_qubit_probability_read_1_given_0, per_qubit_probability_read_0_given_1)
NOISE_MODELS = {
    "raw table": confusion_matrix,
    "learned model": learned_confusion_matrix,
    "independent": independent_confusion_matrix,
}
print()
print("mean absolute entry error versus the true matrix:")
for model_name, noise_matrix in NOISE_MODELS.items():
    print(f"  {model_name:14s} {np.abs(noise_matrix - true_confusion_matrix).mean():.4f}")

In [ ]:
# Test circuits. Each has a known ideal distribution from an exact statevector. Readout noise is the only noise.

def build_ghz_circuit():
    """GHZ state: equal superposition of all zeros and all ones"""
    circuit = QuantumCircuit(QUBIT_COUNT, QUBIT_COUNT)
    circuit.h(0)
    for qubit_index in range(1, QUBIT_COUNT):
        circuit.cx(qubit_index - 1, qubit_index)
    return circuit


def build_spread_circuit():
    """Shallow circuit with a spread-out output distribution"""
    circuit = QuantumCircuit(QUBIT_COUNT, QUBIT_COUNT)
    for qubit_index in range(QUBIT_COUNT):
        circuit.ry(0.6 * (qubit_index + 1), qubit_index)
    for qubit_index in range(QUBIT_COUNT - 1):
        circuit.cx(qubit_index, qubit_index + 1)
    return circuit


def build_random_circuit():
    """Random single-qubit rotations and a ring of CX gates"""
    circuit = QuantumCircuit(QUBIT_COUNT, QUBIT_COUNT)
    for qubit_index in range(QUBIT_COUNT):
        circuit.u(random_generator.uniform(0, np.pi), random_generator.uniform(0, 2 * np.pi),
                  random_generator.uniform(0, 2 * np.pi), qubit_index)
    for qubit_index in range(QUBIT_COUNT):
        circuit.cx(qubit_index, (qubit_index + 1) % QUBIT_COUNT)
    return circuit


test_circuits = {
    "GHZ": build_ghz_circuit(),
    "spread": build_spread_circuit(),
    "random": build_random_circuit(),
}

ideal_distributions = {}
for circuit_name, circuit in test_circuits.items():
    ideal_distributions[circuit_name] = Statevector(circuit).probabilities()   # before the measure gates
    circuit.measure(range(QUBIT_COUNT), range(QUBIT_COUNT))

print(test_circuits["GHZ"].draw(fold=120))
print("ideal GHZ distribution:", np.round(ideal_distributions["GHZ"], 3))

In [ ]:
# Evaluation: run each test circuit through the noisy readout, mitigate with each noise model, compare to ideal.

METHOD_NAMES = ["unmitigated"] + list(NOISE_MODELS.keys())
results = {}
for circuit_name, circuit in test_circuits.items():
    counts = noisy_simulator.run(circuit, shots=TEST_SHOTS).result().get_counts()
    observed = counts_to_probability_vector(counts, TEST_SHOTS)
    ideal = ideal_distributions[circuit_name]
    results[circuit_name] = {"observed": observed, "ideal": ideal, "unmitigated": total_variation_distance(observed, ideal)}
    for model_name, noise_matrix in NOISE_MODELS.items():
        results[circuit_name][model_name] = total_variation_distance(mitigate_by_inversion(observed, noise_matrix), ideal)

print("TOTAL VARIATION DISTANCE TO THE IDEAL DISTRIBUTION (lower is better)")
print(f"{'circuit':10s}" + "".join(f"{method_name:>15s}" for method_name in METHOD_NAMES))
for circuit_name in test_circuits:
    print(f"{circuit_name:10s}" + "".join(f"{results[circuit_name][method_name]:15.4f}" for method_name in METHOD_NAMES))
print(f"{'mean':10s}" + "".join(
    f"{np.mean([results[circuit_name][method_name] for circuit_name in test_circuits]):15.4f}" for method_name in METHOD_NAMES))
print()
print("This is ONE calibration realization and ONE test sample per circuit. Rerun with other seeds before ranking the models.")

config = {
    "RANDOM_SEED": RANDOM_SEED,
    "QUBIT_COUNT": QUBIT_COUNT,
    "PROBABILITY_READ_1_GIVEN_0_LOW": PROBABILITY_READ_1_GIVEN_0_LOW,
    "PROBABILITY_READ_1_GIVEN_0_HIGH": PROBABILITY_READ_1_GIVEN_0_HIGH,
    "PROBABILITY_READ_0_GIVEN_1_LOW": PROBABILITY_READ_0_GIVEN_1_LOW,
    "PROBABILITY_READ_0_GIVEN_1_HIGH": PROBABILITY_READ_0_GIVEN_1_HIGH,
    "CALIBRATION_SHOTS_PER_STATE": CALIBRATION_SHOTS_PER_STATE,
    "TEST_SHOTS": TEST_SHOTS,
    "LOGISTIC_REGRESSION_MAX_ITERATIONS": LOGISTIC_REGRESSION_MAX_ITERATIONS,
    "LOGISTIC_REGRESSION_REGULARIZATION": LOGISTIC_REGRESSION_REGULARIZATION,
    "package_versions": package_versions,
}
dataset = {
    "config": config,
    "true_probability_read_1_given_0": per_qubit_probability_read_1_given_0,
    "true_probability_read_0_given_1": per_qubit_probability_read_0_given_1,
    "calibration_counts": calibration_counts,
    "confusion_matrix": confusion_matrix,
    "learned_confusion_matrix": learned_confusion_matrix,
    "independent_confusion_matrix": independent_confusion_matrix,
    "true_confusion_matrix": true_confusion_matrix,
    "results": results,
}
with open(OUTPUT_PATH, "wb") as output_file:
    pickle.dump(dataset, output_file)
print("saved", OUTPUT_PATH)

In [ ]:
# Poster figure: grouped bars, one group per test circuit, one bar per method.

circuit_names = list(test_circuits.keys())
bar_width = 0.2
group_positions = np.arange(len(circuit_names))

figure, axis = plt.subplots(figsize=(9, 4))
for method_position, method_name in enumerate(METHOD_NAMES):
    heights = [results[circuit_name][method_name] for circuit_name in circuit_names]
    axis.bar(group_positions + method_position * bar_width, heights, bar_width, label=method_name)
axis.set_xticks(group_positions + 1.5 * bar_width)
axis.set_xticklabels(circuit_names)
axis.set_ylabel("total variation distance to ideal")
axis.set_title(f"Readout mitigation on {QUBIT_COUNT} qubits, {CALIBRATION_SHOTS_PER_STATE} calibration shots per state, one realization")
axis.legend()
figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=200)
plt.show()
print("saved", FIGURE_PATH)

## Shred-outs

The base run compares three models of the same readout noise at one small calibration budget, on noise that really is independent. The interesting territory is where the independent model is wrong, where the raw table becomes impossible, and where the budget changes the ranking. Pick one direction below, or invent your own. Each is written as the question your poster answers and the number that answers it. If your extension sweeps a setting, choose it on calibration data held out from the calibration shots you fit on, not on the test circuits.

1. **Correlated readout noise.** Do not add a multi-qubit `ReadoutError` to the Aer noise model: it attaches only to a multi-qubit measure instruction, which this circuit does not have, and it is silently ignored (the noise-model cell explains). Instead, work with the classical assignment matrix directly in numpy. Build a 16x16 column-stochastic matrix M in which a flip on qubit 0 raises the flip probability on qubit 1 (write M as a product of the independent matrix and a conditional pair table, or write the pair table by hand), check with a deterministic matrix that your sampler obeys it, then sample calibration counts with `random_generator.multinomial(shots, M[:, prepared])` and test observations with `random_generator.multinomial(shots, M @ ideal_probabilities)`. The independent model cannot represent M. The learned model's log-probabilities are linear in the prepared bits, so it cannot represent a pair interaction either, unless you add the product of the two prepared bits as a feature. Metric: total variation distance for each model as the correlation strength grows, with and without the interaction feature.

2. **Scaling past the calibration wall.** Calibration needs 2^n circuits. At 8 qubits that is 256, at 12 it is 4096. The independent model only needs each qubit to be prepared in 0 and in 1 a few hundred times, which two circuits (all zeros, all ones) already provide. The learned model can be factorized the same way: one small classifier per qubit. Metric: total variation distance versus qubit count at a fixed total calibration budget, for each model that still fits.

3. **Drift detection.** Calibrate once, then shift the true error rates by a small amount and run the test circuits again without recalibrating. How fast does mitigation quality degrade, and can a cheap monitor (one extra circuit per run) detect that the calibration has gone stale? Metric: total variation distance versus drift size, and the drift size at which your monitor fires.

4. **Calibration budget.** Sweep `CALIBRATION_SHOTS_PER_STATE` from 10 to 10000 and repeat each setting over several seeds. Where is the knee for each model? Where do they tie? Metric: total variation distance versus calibration shots with error bars across seeds, all three models on one plot, with the mean absolute entry error versus the true matrix as a second panel.

5. **A better inversion step.** Least squares with clipping is crude. Replace it with iterative Bayesian unfolding: start from a uniform estimate of the true distribution, compute the posterior over true states for each observed state under the current estimate, average it over the observed shots, repeat until the estimate stops moving. Apply it to all three noise models. Metric: total variation distance before and after, especially on the GHZ circuit where the true distribution is far from uniform.

6. **Mitigate an observable, not a distribution.** Most algorithms need one expectation value, not the whole distribution. Measure the GHZ parity (the product of all Z values) through noisy readout and mitigate that single number. Metric: absolute error in the expectation value for each model, and how it scales with qubit count.

Whatever you choose, load `readout_classifier_dataset.pkl` for the base-run calibration data, all three noise matrices, the true matrix, and the results. The `config` dict inside tells you exactly which run produced it.

In [ ]:
# Your extension starts here.
